In [2]:
import pandas as pd

captains = pd.read_csv(r'C:\Users\AVINASH\Downloads\DataScience Assessment\captains.csv')
doc_events = pd.read_csv(r'C:\Users\AVINASH\Downloads\DataScience Assessment\doc_events.csv')
approvals = pd.read_csv(r'C:\Users\AVINASH\Downloads\DataScience Assessment\approvals.csv')

# Merge datasets
df = captains.merge(approvals, on='captain_id', how='left')

# Let's check dropped_in_docs breakdown by last_stage_reached
print("--- Last stage reached for dropped_in_docs ---")
print(df[df['final_status'] == 'dropped_in_docs']['last_stage_reached'].value_counts())

# Let's check failures by doc_type and failure_reason
failures = doc_events[doc_events['event_type'] == 'verification_fail']
print("\n--- Failures by doc_type and failure_reason ---")
print(pd.crosstab(failures['doc_type'], failures['failure_reason'], margins=True))

# Let's check drop-offs / approval rates across device tiers and vehicle types
print("\n--- Approval Rate by Device Tier ---")
print(df.groupby('device_tier').agg(
    total=('captain_id', 'count'),
    approved=('final_status', lambda x: (x == 'approved').sum()),
    dropped_docs=('final_status', lambda x: (x == 'dropped_in_docs').sum())
).assign(approval_rate=lambda x: x['approved']/x['total']*100, drop_rate=lambda x: x['dropped_docs']/x['total']*100))

print("\n--- Failure reasons by device tier ---")
fail_merged = failures.merge(captains[['captain_id', 'device_tier']], on='captain_id')
print(pd.crosstab(fail_merged['device_tier'], fail_merged['failure_reason'], normalize='index') * 100)

--- Last stage reached for dropped_in_docs ---
last_stage_reached
RC           5720
PERMIT       3385
FITNESS      2908
DL           2823
INSURANCE    2612
AADHAAR      1614
Name: count, dtype: int64

--- Failures by doc_type and failure_reason ---
failure_reason  details_not_legible  document_expired  duplicate_document  \
doc_type                                                                    
AADHAAR                         164               233                 107   
DL                              428               531                 239   
FITNESS                         281               301                 150   
INSURANCE                       272               244                 106   
PERMIT                          189               467                 141   
RC                              837               910                 453   
All                            2171              2686                1196   

failure_reason  image_blurred  name_mismatch  ocr_low_con

In [3]:

rc_failures = doc_events[(doc_events['doc_type'] == 'RC') & (doc_events['event_type'] == 'verification_fail')]
rc_fail_merged = rc_failures.merge(captains, on='captain_id')
print("RC Failures by Device Tier & Reason:")
print(pd.crosstab(rc_fail_merged['device_tier'], rc_fail_merged['failure_reason'], margins=True))
tech_failures = doc_events[
    (doc_events['event_type'] == 'verification_fail') & 
    (doc_events['failure_reason'].isin(['image_blurred', 'ocr_low_confidence', 'details_not_legible']))
]
print("\nUnique captains with technical/image quality failures:", tech_failures['captain_id'].nunique())

dropped_tech = tech_failures.merge(approvals, on='captain_id')
print("Dropped in docs among technical failures:", (dropped_tech['final_status'] == 'dropped_in_docs').sum())

RC Failures by Device Tier & Reason:
failure_reason  details_not_legible  document_expired  duplicate_document  \
device_tier                                                                 
high                             71               107                  62   
low                             475               406                 191   
mid                             291               397                 200   
All                             837               910                 453   

failure_reason  image_blurred  name_mismatch  ocr_low_confidence  \
device_tier                                                        
high                      142            134                  93   
low                      1709            504                1332   
mid                       437            486                 348   
All                      2288           1124                1773   

failure_reason  wrong_document_type   All  
device_tier                                
hig